# Ex 1

Implement ResNet3D-18 from scratch starting from the implementation of ResNet in *Laboratory 4*.

In [ ]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 19.2 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import random_split, DataLoader
from torch.optim.lr_scheduler import StepLR
import torchvision
from torchvision.models.video import r3d_18
from torchvision import transforms
import os
import warnings

torch.manual_seed(42);

warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p video_data test_train_splits
!unrar e 'drive/MyDrive/PI/test_train_splits.rar' test_train_splits # Provide the proper path location
!unrar e 'drive/MyDrive/PI/hmdb51_org.rar' video_data # Provide the proper path location

for files in os.listdir('video_data'):
    foldername = files.split('.')[0]
    os.system("mkdir -p video_data/" + foldername)
    os.system("unrar e video_data/"+ files + " video_data/" + foldername)
!rm video_data/*.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from drive/MyDrive/PI/test_train_splits.rar

Extracting  test_train_splits/brush_hair_test_split1.txt                   0%  OK 
Extracting  test_train_splits/brush_hair_test_split2.txt                   1%  OK 
Extracting  test_train_splits/brush_hair_test_split3.txt                   1%  OK 
Extracting  test_train_splits/cartwheel_test_split1.txt                    2%  OK 
Extracting  test_train_splits/cartwheel_test_split2.txt                    2%  OK 
Extracting  test_train_splits/cartwheel_test_split3.txt                    3%  OK 
Extracting  test_train_splits/catch_test_split1.txt                        4%  OK 
Extracting  test_train_splits/catch_test_split2.txt                        4%  OK 
Extracting  test_train_splits/catch_test_split3.txt                        5%  OK 
Extracting  test_train_splits/chew_test_

In [ ]:
val_split = 0.05 #3 min
num_frames = 16
clip_steps = 50
num_workers = 8
bs = 4

train_tfms = torchvision.transforms.Compose([
                                 transforms.Lambda(lambda x: x.permute(3, 0, 1, 2).to(torch.float32) / 255.),
                                 transforms.Resize((128, 171)),
                                 transforms.RandomHorizontalFlip(),
                                 transforms.RandomCrop((112, 112))
                               ])
test_tfms = torchvision.transforms.Compose([
                                             transforms.Lambda(lambda x: x.permute(3, 0, 1, 2).to(torch.float32) / 255.),
                                             transforms.Resize((128, 171)),
                                             transforms.CenterCrop((112, 112))
                                             ])

hmdb51_train = torchvision.datasets.HMDB51('video_data/', 'test_train_splits/', num_frames,
                                                step_between_clips=clip_steps, fold=1, train=True,
                                                transform=train_tfms, num_workers=num_workers)

hmdb51_test = torchvision.datasets.HMDB51('video_data/', 'test_train_splits/', num_frames,
                                                step_between_clips=clip_steps, fold=1, train=False,
                                                transform=test_tfms, num_workers=num_workers)


total_train_samples = len(hmdb51_train)
total_val_samples = round(val_split * total_train_samples)

hmdb51_train_v1, hmdb51_val_v1 = random_split(hmdb51_train, [total_train_samples - total_val_samples, total_val_samples])

train_loader = DataLoader(hmdb51_train_v1, batch_size=bs, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(hmdb51_val_v1, batch_size=bs, shuffle=True, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(hmdb51_test, batch_size=bs, shuffle=False, num_workers=num_workers, pin_memory=True)

print(f"number of train samples {total_train_samples - total_val_samples}")
print(f"number of validation samples {total_val_samples}")
print(f"number of test samples {len(hmdb51_test)}")

100%|██████████| 423/423 [01:41<00:00,  4.17it/s]


number of train samples 7366
number of validation samples 388
number of test samples 3234


In [ ]:
class VideoRecognitionModel(nn.Module):
  def __init__(self):
      super(VideoRecognitionModel, self).__init__()
      self.base_model = nn.Sequential(*list(r3d_18().children())[:-1])
      self.fc = nn.Linear(512, 51)

  def forward(self, x):
      out = self.base_model(x).flatten(1)
      out = F.relu(self.fc(out))
      return out

In [ ]:
def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

In [ ]:
def train(model, loader, optimizer, epoch, device=try_gpu()):
    model.train()
    model = model.to(device)
    total_loss, total_correct, num_labels = 0, 0, 0
    for batch_id, data in enumerate(loader):
        data, target = data[0], data[-1]

        data = data.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_labels += data.size(0)

        pred = output.argmax(dim=1, keepdim=True)  # get the index of the max probability
        num_corrects = pred.eq(target.view_as(pred)).sum().item()

        total_correct += num_corrects

        if (batch_id + 1) % 100 == 0:
           print('Train Epoch: {} Batch [{}/{} ({:.0f}%)]\tLoss: {:.6f} Accuracy: {}/{} ({:.0f}%)'.format(
                epoch, (batch_id + 1) * len(data), len(loader.dataset),
                100. * (batch_id + 1) / len(loader), total_loss / num_labels, total_correct, num_labels, 100. * total_correct / num_labels))

    print('Train Epoch: {} Average Loss: {:.6f} Average Accuracy: {}/{} ({:.0f}%)'.format(
         epoch, total_loss / num_labels, total_correct, num_labels, 100. * total_correct / num_labels))

In [ ]:
def test(model, loader, text='Validation', device=try_gpu()):
    model.eval()
    model = model.to(device)
    total_loss, total_correct, num_labels = 0, 0, 0
    with torch.no_grad():
         for batch_id, data in enumerate(loader):
             data, target = data[0], data[-1]

             data = data.to(device)
             target = target.to(device)

             output = model(data)
             loss = F.cross_entropy(output, target)

             total_loss += loss.item()
             num_labels += data.size(0)

             pred = output.argmax(dim=1, keepdim=True)  # get the index of the max probability
             num_corrects = pred.eq(target.view_as(pred)).sum().item()

             total_correct += num_corrects

    print(text + ' Average Loss: {:.6f} Average Accuracy: {}/{} ({:.0f}%)'.format(
         total_loss / num_labels, total_correct, num_labels, 100. * total_correct / num_labels))

In [ ]:
class Residual3D(nn.Module):
    """The Residual block of ResNet."""
    def __init__(self, input_channels, num_channels,
                 use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv3d(input_channels, num_channels,
                               kernel_size=3, padding=1, stride=strides)
        self.bn1 = nn.BatchNorm3d(num_channels)

        self.conv2 = nn.Conv3d(num_channels, num_channels,
                               kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(num_channels)

        if use_1x1conv:
            self.conv3 = nn.Conv3d(input_channels, num_channels,
                                   kernel_size=1, stride=strides)
        else:
            self.conv3 = None

    def forward(self, X):
        Y = nn.ReLU()(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return nn.ReLU()(Y)

In [ ]:
def resnet_block_3d(input_channels, num_channels, num_residuals,
                 first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual3D(input_channels, num_channels,
                                use_1x1conv=True, strides=2))
        else:
            blk.append(Residual3D(num_channels, num_channels))
    return blk

## V1 - blocuri ca la lab

In [ ]:
b1 = nn.Sequential(
    nn.Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3)),
    nn.BatchNorm3d(64),
    nn.ReLU(),
    nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
)

In [ ]:
b2 = nn.Sequential(*resnet_block_3d(64, 64, 2, first_block=True))
b3 = nn.Sequential(*resnet_block_3d(64, 128, 2))
b4 = nn.Sequential(*resnet_block_3d(128, 256, 2))
b5 = nn.Sequential(*resnet_block_3d(256, 512, 2))

In [ ]:
net = nn.Sequential(b1, b2, b3, b4, b5,
                    nn.AdaptiveAvgPool3d((1, 1, 1)), # Changed to AdaptiveAvgPool3d
                    nn.Flatten(), nn.Linear(512, 51)) # Changed output features to 51 classes
net

Sequential(
  (0): Sequential(
    (0): Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3))
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), dilation=1, ceil_mode=False)
  )
  (1): Sequential(
    (0): Residual3D(
      (conv1): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (bn1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (bn2): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): Residual3D(
      (conv1): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (bn1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv3d(64, 64, kernel_size=(3, 3, 3), 

In [ ]:
lr = 1e-2 #9 min
gamma = 0.7
total_epochs = 1

model = net # Using the 'net' model as requested

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

for epoch in range(1, total_epochs + 1):
    train(model, train_loader, optimizer, epoch)
    test(model, val_loader, text="Validation")
    scheduler.step()

Train Epoch: 1 Batch [400/7366 (5%)]	Loss: 1.159205 Accuracy: 9/400 (2%)
Train Epoch: 1 Batch [800/7366 (11%)]	Loss: 1.068568 Accuracy: 23/800 (3%)
Train Epoch: 1 Batch [1200/7366 (16%)]	Loss: 1.038147 Accuracy: 40/1200 (3%)
Train Epoch: 1 Batch [1600/7366 (22%)]	Loss: 1.021292 Accuracy: 54/1600 (3%)
Train Epoch: 1 Batch [2000/7366 (27%)]	Loss: 1.012011 Accuracy: 66/2000 (3%)
Train Epoch: 1 Batch [2400/7366 (33%)]	Loss: 1.003841 Accuracy: 88/2400 (4%)
Train Epoch: 1 Batch [2800/7366 (38%)]	Loss: 1.000668 Accuracy: 104/2800 (4%)
Train Epoch: 1 Batch [3200/7366 (43%)]	Loss: 0.996570 Accuracy: 118/3200 (4%)
Train Epoch: 1 Batch [3600/7366 (49%)]	Loss: 0.992034 Accuracy: 145/3600 (4%)
Train Epoch: 1 Batch [4000/7366 (54%)]	Loss: 0.990058 Accuracy: 160/4000 (4%)
Train Epoch: 1 Batch [4400/7366 (60%)]	Loss: 0.986744 Accuracy: 180/4400 (4%)
Train Epoch: 1 Batch [4800/7366 (65%)]	Loss: 0.984880 Accuracy: 201/4800 (4%)
Train Epoch: 1 Batch [5200/7366 (71%)]	Loss: 0.983416 Accuracy: 222/5200 (4%

In [ ]:
test(model, test_loader, text="Test") #2 min

Test Average Loss: 0.967290 Average Accuracy: 202/3234 (6%)


## V2 - clasa

In [ ]:
class ResNet3D_18(nn.Module):
    def __init__(self, num_classes=51):
        super().__init__()

        # b1: "Stem"-ul rețelei.
        # În Lab 4 era Conv2d(1, ...). Aici punem Conv3d(3, ...) pentru RGB.
        # Folosim un kernel (3, 7, 7) tipic pentru video (timp mic, spațiu mare).
        self.b1 = nn.Sequential(
            nn.Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3)),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        )

        # b2 - b5: Blocurile ResNet standard (2 blocuri per stage pentru ResNet18)
        self.b2 = nn.Sequential(*resnet_block_3d(64, 64, 2, first_block=True))
        self.b3 = nn.Sequential(*resnet_block_3d(64, 128, 2))
        self.b4 = nn.Sequential(*resnet_block_3d(128, 256, 2))
        self.b5 = nn.Sequential(*resnet_block_3d(256, 512, 2))

        # Head-ul de clasificare
        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        x = self.b5(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

In [ ]:
lr = 1e-2 #9 min
gamma = 0.7
total_epochs = 1

model = ResNet3D_18(num_classes = 51)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

for epoch in range(1, total_epochs + 1):
    train(model, train_loader, optimizer, epoch)
    test(model, val_loader, text="Validation")
    scheduler.step()

Train Epoch: 1 Batch [400/7366 (5%)]	Loss: 1.155772 Accuracy: 11/400 (3%)
Train Epoch: 1 Batch [800/7366 (11%)]	Loss: 1.065406 Accuracy: 28/800 (4%)
Train Epoch: 1 Batch [1200/7366 (16%)]	Loss: 1.039558 Accuracy: 48/1200 (4%)
Train Epoch: 1 Batch [1600/7366 (22%)]	Loss: 1.024245 Accuracy: 64/1600 (4%)
Train Epoch: 1 Batch [2000/7366 (27%)]	Loss: 1.013817 Accuracy: 85/2000 (4%)
Train Epoch: 1 Batch [2400/7366 (33%)]	Loss: 1.008123 Accuracy: 101/2400 (4%)
Train Epoch: 1 Batch [2800/7366 (38%)]	Loss: 1.001454 Accuracy: 120/2800 (4%)
Train Epoch: 1 Batch [3200/7366 (43%)]	Loss: 0.997633 Accuracy: 137/3200 (4%)
Train Epoch: 1 Batch [3600/7366 (49%)]	Loss: 0.994330 Accuracy: 159/3600 (4%)
Train Epoch: 1 Batch [4000/7366 (54%)]	Loss: 0.990235 Accuracy: 173/4000 (4%)
Train Epoch: 1 Batch [4400/7366 (60%)]	Loss: 0.987072 Accuracy: 200/4400 (5%)
Train Epoch: 1 Batch [4800/7366 (65%)]	Loss: 0.985511 Accuracy: 215/4800 (4%)
Train Epoch: 1 Batch [5200/7366 (71%)]	Loss: 0.983591 Accuracy: 233/5200 (

In [ ]:
test(model, test_loader, text="Test") #2 min

Test Average Loss: 0.953543 Average Accuracy: 243/3234 (8%)


# Ex 2

Replace the ResNet3D-18 model with the C3D model given in *Figure 8, Lecture 9*, and compare their performance.

In [ ]:
import torch
import torch.nn as nn

class C3D(nn.Module):
    def __init__(self, num_classes=51):
        super(C3D, self).__init__()

        # --- Conv1 Block ---
        # 64 filtre, input 3 canale (RGB)
        self.conv1 = nn.Conv3d(3, 64, kernel_size=3, padding=1)
        # Pool1: kernel 1x2x2 -> reduce doar H și W, nu și Timpul
        self.pool1 = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

        # --- Conv2 Block ---
        # 128 filtre
        self.conv2 = nn.Conv3d(64, 128, kernel_size=3, padding=1)
        # Pool2: kernel 2x2x2
        self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)

        # --- Conv3 Block ---
        # 256 filtre, repetat de 2 ori (3a, 3b)
        self.conv3a = nn.Conv3d(128, 256, kernel_size=3, padding=1)
        self.conv3b = nn.Conv3d(256, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2)

        # --- Conv4 Block ---
        # 512 filtre, repetat de 2 ori (4a, 4b)
        self.conv4a = nn.Conv3d(256, 512, kernel_size=3, padding=1)
        self.conv4b = nn.Conv3d(512, 512, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool3d(kernel_size=2, stride=2)

        # --- Conv5 Block ---
        # 512 filtre, repetat de 2 ori (5a, 5b)
        self.conv5a = nn.Conv3d(512, 512, kernel_size=3, padding=1)
        self.conv5b = nn.Conv3d(512, 512, kernel_size=3, padding=1)
        self.pool5 = nn.MaxPool3d(kernel_size=2, stride=2)

        # --- Fully Connected Layers ---
        # Calculăm dimensiunea intrării în FC:
        # Input video: (Batch, 3, 16, 112, 112)
        # Pool1 -> (16, 56, 56)
        # Pool2 -> (8, 28, 28)
        # Pool3 -> (4, 14, 14)
        # Pool4 -> (2, 7, 7)
        # Pool5 -> (1, 3, 3)  <-- (7 // 2 = 3)
        # Total Features: 512 canale * 1 * 3 * 3 = 4608

        self.fc6 = nn.Linear(4608, 4096)
        self.fc7 = nn.Linear(4096, 4096)
        self.fc8 = nn.Linear(4096, num_classes)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5) # Uzual pentru C3D/VGG

    def forward(self, x):
        # Block 1
        x = self.relu(self.conv1(x))
        x = self.pool1(x)

        # Block 2
        x = self.relu(self.conv2(x))
        x = self.pool2(x)

        # Block 3
        x = self.relu(self.conv3a(x))
        x = self.relu(self.conv3b(x))
        x = self.pool3(x)

        # Block 4
        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x))
        x = self.pool4(x)

        # Block 5
        x = self.relu(self.conv5a(x))
        x = self.relu(self.conv5b(x))
        x = self.pool5(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # FC Layers
        x = self.relu(self.fc6(x))
        x = self.dropout(x)
        x = self.relu(self.fc7(x))
        x = self.dropout(x)

        # Output
        x = self.fc8(x)

        return x

In [ ]:
lr = 1e-2 #9 min
gamma = 0.7
total_epochs = 1

model = C3D(num_classes=51)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

for epoch in range(1, total_epochs + 1):
    train(model, train_loader, optimizer, epoch)
    test(model, val_loader, text="Validation")
    scheduler.step()

Train Epoch: 1 Batch [400/7366 (5%)]	Loss: 403636048.177343 Accuracy: 9/400 (2%)
Train Epoch: 1 Batch [800/7366 (11%)]	Loss: 201819182.214271 Accuracy: 15/800 (2%)
Train Epoch: 1 Batch [1200/7366 (16%)]	Loss: 134546121.816902 Accuracy: 28/1200 (2%)
Train Epoch: 1 Batch [1600/7366 (22%)]	Loss: 100909591.613669 Accuracy: 39/1600 (2%)
Train Epoch: 1 Batch [2000/7366 (27%)]	Loss: 80727673.487810 Accuracy: 53/2000 (3%)
Train Epoch: 1 Batch [2400/7366 (33%)]	Loss: 67273061.404690 Accuracy: 68/2400 (3%)
Train Epoch: 1 Batch [2800/7366 (38%)]	Loss: 57662624.202102 Accuracy: 84/2800 (3%)
Train Epoch: 1 Batch [3200/7366 (43%)]	Loss: 50454796.299610 Accuracy: 107/3200 (3%)
Train Epoch: 1 Batch [3600/7366 (49%)]	Loss: 44848707.930947 Accuracy: 133/3600 (4%)
Train Epoch: 1 Batch [4000/7366 (54%)]	Loss: 40363837.235635 Accuracy: 145/4000 (4%)
Train Epoch: 1 Batch [4400/7366 (60%)]	Loss: 36694397.576891 Accuracy: 161/4400 (4%)
Train Epoch: 1 Batch [4800/7366 (65%)]	Loss: 33636531.193767 Accuracy: 184

In [ ]:
test(model, test_loader, text="Test") #2 min

Test Average Loss: 0.972326 Average Accuracy: 179/3234 (6%)
